# Session 04 Practice: Informed Search with A* Routing & Heuristic Benchmarking
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 2 | Session 4**  

---

### 🎯 Learning Objectives
1. Formulate the **A* Search Algorithm** with an admissible and consistent heuristic $h(n)$.
2. Design the **Euclidean Travel Time Heuristic**: $h(n) = \frac{\text{dist}(n, \text{goal})}{v_{\max}}$.
3. Compare search frontier expansions between **Dijkstra** (uniform cost) and **A*** (informed search).
4. Benchmark search performance (explored nodes, runtime, optimality) using visual comparison plots in **Plotly**.

---

### 📐 Mathematical Formulation

$$f(n) = g(n) + h(n)$$

Where:
- $g(n)$: Exact cost-to-come from start node to node $n$.
- $h(n)$: Estimated cost-to-go from node $n$ to goal.
- **Admissibility Condition:** $0 \le h(n) \le h^*(n)$ (never overestimates true remaining cost).
- **Consistency (Monotonicity):** $h(u) \le w(u, v) + h(v)$ for all edges $(u, v)$.


In [ ]:
import heapq
import math
import time
import networkx as nx
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Urban Grid Road Network Builder

In [ ]:
def create_grid_road_network(rows: int = 15, cols: int = 15, spacing: float = 50.0) -> nx.DiGraph:
    G = nx.DiGraph()
    for r in range(rows):
        for c in range(cols):
            node_id = r * cols + c
            x = c * spacing
            y = r * spacing
            G.add_node(node_id, pos=(x, y))
            
    for r in range(rows):
        for c in range(cols):
            u = r * cols + c
            if c + 1 < cols:
                v = r * cols + (c + 1)
                speed = 20.0 if (r % 3 == 0) else 10.0 # Arterial vs local street
                dist = spacing
                G.add_edge(u, v, length=dist, speed=speed, weight=dist/speed)
                G.add_edge(v, u, length=dist, speed=speed, weight=dist/speed)
            if r + 1 < rows:
                v = (r + 1) * cols + c
                speed = 20.0 if (c % 3 == 0) else 10.0
                dist = spacing
                G.add_edge(u, v, length=dist, speed=speed, weight=dist/speed)
                G.add_edge(v, u, length=dist, speed=speed, weight=dist/speed)
                
    return G

G = create_grid_road_network(rows=15, cols=15, spacing=40.0)
print(f"Urban grid generated: {len(G.nodes)} intersections, {len(G.edges)} road links.")


## 2. A* and Dijkstra Search Implementations

In [ ]:
def euclidean_travel_time_heuristic(graph: nx.DiGraph, node: int, goal: int, max_speed: float = 20.0) -> float:
    x1, y1 = graph.nodes[node]['pos']
    x2, y2 = graph.nodes[goal]['pos']
    dist = math.hypot(x2 - x1, y2 - y1)
    return dist / max_speed

def astar_search(graph: nx.DiGraph, start_node: int, goal_node: int, max_speed: float = 20.0, weight_h: float = 1.0):
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    
    h_start = euclidean_travel_time_heuristic(graph, start_node, goal_node, max_speed)
    pq = [(weight_h * h_start, 0.0, start_node)]
    parent = {node: None for node in graph.nodes}
    
    visited = set()
    explored_order = []
    
    while pq:
        f, current_g, u = heapq.heappop(pq)
        
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        if u == goal_node:
            path = []
            curr = goal_node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return path, current_g, len(explored_order), explored_order
            
        for v in graph.neighbors(u):
            w = graph[u][v]['weight']
            tentative_g = current_g + w
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                h_v = euclidean_travel_time_heuristic(graph, v, goal_node, max_speed)
                heapq.heappush(pq, (tentative_g + weight_h * h_v, tentative_g, v))
                
    return [], float('inf'), len(explored_order), explored_order

def dijkstra_search(graph: nx.DiGraph, start_node: int, goal_node: int):
    return astar_search(graph, start_node, goal_node, weight_h=0.0)


## 3. Comparative Frontier Visualization & Benchmark Dashboard

In [ ]:
start_id = 0
goal_id = len(G.nodes) - 1

# Execute both searches
t0 = time.perf_counter()
d_path, d_cost, d_exp, d_order = dijkstra_search(G, start_id, goal_id)
t_dijkstra = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
a_path, a_cost, a_exp, a_order = astar_search(G, start_id, goal_id, max_speed=20.0, weight_h=1.0)
t_astar = (time.perf_counter() - t0) * 1000

print(f"Dijkstra: {d_exp} nodes explored ({d_exp/len(G.nodes)*100:.1f}%), travel time = {d_cost:.2f}s")
print(f"A* Search: {a_exp} nodes explored ({a_exp/len(G.nodes)*100:.1f}%), travel time = {a_cost:.2f}s")


In [ ]:
# Side-by-Side Frontier Exploration Map
fig_comp = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Dijkstra Search (Explored: {d_exp} nodes)",
        f"A* Informed Search (Explored: {a_exp} nodes)"
    )
)

# 1. Dijkstra Explored Heatmap
node_x = [G.nodes[n]['pos'][0] for n in G.nodes]
node_y = [G.nodes[n]['pos'][1] for n in G.nodes]
d_colors = [d_order.index(n) if n in d_order else -1 for n in G.nodes]

fig_comp.add_trace(
    go.Scatter(x=node_x, y=node_y, mode="markers",
               marker=dict(size=8, color=d_colors, colorscale="Viridis", showscale=False),
               name="Dijkstra Frontier"),
    row=1, col=1
)
# Dijkstra path
d_path_x = [G.nodes[n]['pos'][0] for n in d_path]
d_path_y = [G.nodes[n]['pos'][1] for n in d_path]
fig_comp.add_trace(
    go.Scatter(x=d_path_x, y=d_path_y, mode="lines+markers",
               line=dict(color="red", width=4), name="Dijkstra Path"),
    row=1, col=1
)

# 2. A* Explored Heatmap
a_colors = [a_order.index(n) if n in a_order else -1 for n in G.nodes]
fig_comp.add_trace(
    go.Scatter(x=node_x, y=node_y, mode="markers",
               marker=dict(size=8, color=a_colors, colorscale="Viridis", showscale=True, colorbar=dict(title="Visit Order")),
               name="A* Frontier"),
    row=1, col=2
)
# A* path
a_path_x = [G.nodes[n]['pos'][0] for n in a_path]
a_path_y = [G.nodes[n]['pos'][1] for n in a_path]
fig_comp.add_trace(
    go.Scatter(x=a_path_x, y=a_path_y, mode="lines+markers",
               line=dict(color="gold", width=4), name="A* Path"),
    row=1, col=2
)

fig_comp.update_layout(
    title="Search Frontier Comparison: Dijkstra (Blind Radial Expansion) vs A* (Directed Cone)",
    height=600
)
fig_comp.show()


In [ ]:
# Efficiency Metrics Comparison Chart
fig_bars = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Nodes Explored (Lower is Better)", "Search Execution Time (ms)")
)

algorithms = ["Dijkstra (Uniform Cost)", "A* (Euclidean Time Heuristic)"]
explored_counts = [d_exp, a_exp]
runtimes = [t_dijkstra, t_astar]

fig_bars.add_trace(
    go.Bar(x=algorithms, y=explored_counts, marker_color=["#e74c3c", "#2ecc71"], name="Explored Nodes"),
    row=1, col=1
)
fig_bars.add_trace(
    go.Bar(x=algorithms, y=runtimes, marker_color=["#e74c3c", "#2ecc71"], name="Runtime (ms)"),
    row=1, col=2
)

fig_bars.update_layout(title="A* vs Dijkstra Efficiency Metrics", height=420, showlegend=False)
fig_bars.show()


## 4. Verification & Consistency Assertions

In [ ]:
assert math.isclose(d_cost, a_cost, rel_tol=1e-3), f"A* path cost {a_cost} was suboptimal compared to Dijkstra {d_cost}!"
assert a_exp <= d_exp, f"A* explored ({a_exp}) should explore fewer or equal nodes than Dijkstra ({d_exp})!"

print("✅ All A* heuristic search benchmarks and tests passed successfully!")
